# Travel desk V3-R1 — Qwen3-1.7B + LoRA (Google Colab)

**V3-R1 is FROZEN_ACTIVE on main. Run the preflight and data overlap checks before training.** This is the only notebook entry point for new travel-desk training. Historical V2.2 is archived separately; do not use its Golden Test or results for this run. The preflight checks the active release record, committed content and SHA-256 before dataset preparation. The guard verifies the active freeze record and stops if the checkout or data is wrong.

Use a T4 GPU. Fill in the settings, use a fresh clone and distinct adapter ID. Do not publish weights or corpus-derived data until licensing is cleared. No end-to-end V3 run is claimed.


## 1. Settings

In [ ]:
# Set these two values, then run the cell.
REPO_URL = "https://github.com/dhrtt990-blip/saudi-dialect-to-msa.git"      # your team's GitHub repository, e.g. "https://github.com/<team>/<repo>.git"
HF_REPO_ID = ""    # optional for local training; do not publish before licence clearance

BASE_MODEL = "Qwen/Qwen3-1.7B"   # 1.7B parameters: inside the 4B limit of the project
GGUF_NAME = "travel-v3-r1-qwen3-1.7b-q8_0.gguf"
BASE_GGUF_NAME = "travel-v3-r1-qwen3-base-q8_0.gguf"   # the untuned model, converted like the tuned one: the baseline

USE_QLORA = False   # True: load the base model in 4-bit (less GPU memory, slightly slower)
EPOCHS = 2
LEARNING_RATE = 1e-4
MAX_LENGTH = 768    # tokens per training example (system prompt + message + rewrite)
BATCH_SIZE = 4
GRAD_ACCUMULATION = 4
LORA_R = 16
LORA_ALPHA = 32
SEED = 13
CORPUS_CSV = "data/raw/multidialectal_travel_corpus.csv"   # downloaded in step 4

# The corpus has almost no numbers, so the template pairs are what teaches the model to keep facts.
# These two values make about one training pair in five carry facts. They are a starting point, not a tuned choice.
CORPUS_LIMIT = 8000   # cleaned corpus pairs to use (0 = all of them, about 24,000)
SYNTHETIC_N = 2000    # template pairs with numbers, dates, times, codes and names

assert REPO_URL, "Fill in REPO_URL above."
REPO_REF = "66d26070ba6b98ca3356e39914282bfc72326c76"  # use the final freeze commit on main after activation, never a historical commit
GOLDEN_PATH = "data/golden_v3_r1_candidate.jsonl"


## 2. GPU, code and libraries

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
!git clone {REPO_URL} repo
%cd repo
!git checkout {REPO_REF}
!pip install -q -r requirements-train.txt
# Run the environment compatibility cell before importing datasets or PEFT.


In [ ]:
# Colab includes optional quantization tools that are not needed for float16 LoRA.
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","uninstall","-y","torchao"],check=True)
subprocess.run([sys.executable,"-m","pip","install","pyarrow==23.0.1"],check=True)
# On a new runtime run this immediately after package installation, before dataset loading.


## 3. Required active-release preflight

This guard refuses all training until V3-R1 is formally activated on the checked-out repository. Never edit the release manifest locally to bypass it. The final freeze must be committed and reviewed first.


In [ ]:
import subprocess
import sys
import json
import hashlib
from pathlib import Path

manifest_path = Path("review/v3_r1_revision_manifest.json")
assert manifest_path.is_file(), "V3-R1 is not available on this checkout. Do not use V2.2. Use the approved freeze commit after merge."
m = json.loads(manifest_path.read_text(encoding="utf-8"))
assert m.get("status") == "FROZEN_ACTIVE", "V3-R1 is not formally activated. Stop; obtain approval and commit freeze evidence first."
assert m.get("candidate") == GOLDEN_PATH, "Wrong release path"
assert m.get("freeze_commit"), "Missing recorded freeze commit"
assert hashlib.sha256(Path(GOLDEN_PATH).read_bytes()).hexdigest() == m["sha256"], "Wrong golden bytes"
rows = [json.loads(line) for line in Path(GOLDEN_PATH).read_text(encoding="utf-8").splitlines()]
assert [r["id"] for r in rows] == [f"t{i:03d}" for i in range(1, 101)], "Not the travel golden set"
for path in (GOLDEN_PATH, str(manifest_path)):
    assert not subprocess.check_output(["git", "status", "--porcelain", "--", path], text=True).strip(), f"Uncommitted change: {path}"
    subprocess.run(["git", "ls-files", "--error-unmatch", path], check=True, capture_output=True)
subprocess.run(["git", "merge-base", "--is-ancestor", m["freeze_commit"], "HEAD"], check=True)
assert subprocess.check_output(["git", "show", f"{m['freeze_commit']}:{GOLDEN_PATH}"]) == Path(GOLDEN_PATH).read_bytes(), "Freeze commit does not match data"
subprocess.run([sys.executable, "-m", "src.validate_golden", GOLDEN_PATH], check=True)
print("V3-R1 active:", m["sha256"], "freeze:", m["freeze_commit"])
V3_PREFLIGHT_OK = True


## 4. Build the training data

The corpus is "a Multidialectal Parallel Arabic Corpus - LLM" by Khalid Almeman (Zenodo record 17776617; the licence statements are in `docs/DATASET_CARD.md`). The first cell downloads the CSV (18.4 MB) with `scripts/download_corpus.py`, checks its size and MD5 against the values Zenodo publishes, and prints the columns and three pairs. The file has six columns, and every cell of `Saudi Dialect` starts with the label `- Saudi: `, which is removed on loading. If the download is refused, download the CSV by hand from https://zenodo.org/records/17776617, upload it to `data/raw/` and run the cell again: it then checks the uploaded file. The script was written without network access to Zenodo, so this cell is the first time it runs against the real site.

`src.data_prep` then drops repeated sentences and pairs whose MSA side does not carry the same numbers and Latin words as the dialect side, keeps `CORPUS_LIMIT` corpus pairs, adds `SYNTHETIC_N` seeded template pairs, adds any team-written files named `data/team_*.jsonl`, removes every pair that is close to a golden text or shares a distinctive fact with one, and fails if any overlap remains. Download `data/processed_v3_r1/prep_report.json` with the last cell of this step and commit it: it is the proof for the dataset card.

In [ ]:
import subprocess
import sys

result = subprocess.run([sys.executable, "scripts/download_corpus.py", "--out", CORPUS_CSV], capture_output=True, text=True)
print(result.stdout[-3000:], result.stderr[-2000:])
assert result.returncode == 0, "the corpus was not downloaded or is not the expected file; read the message above"

In [ ]:
import glob
import subprocess
import sys

assert globals().get("V3_PREFLIGHT_OK") is True, "Run the V3-R1 preflight successfully first."
cmd = [sys.executable, "-m", "src.data_prep",
       "--golden", GOLDEN_PATH,
       "--corpus-csv", CORPUS_CSV,
       "--corpus-limit", str(CORPUS_LIMIT),
       "--synthetic-n", str(SYNTHETIC_N),
       "--seed", str(SEED),
       "--out", "data/processed_v3_r1"]
team_files = sorted(glob.glob("data/team_*.jsonl"))
if team_files:
    cmd += ["--team-jsonl", *team_files]
print(" ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-4000:], result.stderr[-2000:])
assert result.returncode == 0, "data preparation failed; read the message above"
report = json.loads(Path("data/processed_v3_r1/prep_report.json").read_text(encoding="utf-8"))
assert report.get("golden_sha256") == m["sha256"], "Preparation used the wrong benchmark"
assert report.get("overlap_remaining_after_filter") == 0, "Overlap remains; stop before training"
V3_PREP_OK = True


## 5. Load the data and look at what the model will see

Compare the two printouts. The training example should contain the same turn markers as the inference prompt, including the empty `<think>` block that Qwen3 uses when thinking is off. If the training text has no `<think>` block while the inference prompt has one, tell the team before training: the model would be trained in a slightly different format from the one it is served in.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

ds = load_dataset("json", data_files={"train": "data/processed_v3_r1/train.jsonl",
                                      "validation": "data/processed_v3_r1/val.jsonl"})
ds = ds.select_columns(["messages"])
print(ds)

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
example = ds["train"][0]["messages"]
print("=== one training example as the model sees it ===")
print(tok.apply_chat_template(example, tokenize=False))
print("=== the prompt used at inference (thinking off) ===")
print(tok.apply_chat_template(example[:-1], tokenize=False, add_generation_prompt=True, enable_thinking=False))

sample = ds["train"].select(range(min(500, len(ds["train"]))))["messages"]
lengths = [len(tok(tok.apply_chat_template(m, tokenize=False))["input_ids"]) for m in sample]
too_long = sum(n > MAX_LENGTH for n in lengths) / len(lengths)
print(f"token length: max {max(lengths)}, share above MAX_LENGTH: {too_long:.1%}")

## 6. Train the LoRA adapter

`assistant_only_loss=True` puts the loss on the MSA rewrite only, not on the instruction or the dialect message. If the GPU runs out of memory, set `USE_QLORA = True` in the settings cell, or lower `BATCH_SIZE` and raise `GRAD_ACCUMULATION` by the same factor, then restart the runtime.

In [ ]:
assert globals().get("V3_PREP_OK") is True
assert globals().get("V3_PREFLIGHT_OK") is True
import time, platform, traceback, importlib.metadata
from datetime import datetime, timezone
from peft import LoraConfig
from transformers import TrainerCallback
from trl import SFTConfig, SFTTrainer
RUN_DIR = Path("outputs") / ("travel-v3-r1-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
RUN_DIR.mkdir(parents=True, exist_ok=False)
def save_json(name, value):
    (RUN_DIR / name).write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
versions = {p: importlib.metadata.version(p) for p in ["torch","transformers","peft","trl","datasets","accelerate","huggingface_hub"]}
provenance = {"code_commit": subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip(), "golden_sha256":m["sha256"], "prep_report":report, "seed":SEED, "versions":versions, "python":platform.python_version(), "platform":platform.platform(), "gpu":torch.cuda.get_device_name(0), "gpu_bytes":torch.cuda.get_device_properties(0).total_memory, "started_utc":datetime.now(timezone.utc).isoformat(), "status":"initializing"}
save_json("provenance.json", provenance)
(RUN_DIR / "requirements-train.lock.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True))
class EvidenceCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        save_json("train_log_history.json",state.log_history)
        save_json("progress.json",{"global_step":state.global_step,"max_steps":state.max_steps,"epoch":state.epoch,"updated_utc":datetime.now(timezone.utc).isoformat()})
    def on_step_end(self,args,state,control,**kwargs):
        if state.global_step == 1:
            save_json("first_step.json",{"global_step":state.global_step,"utc":datetime.now(timezone.utc).isoformat()})
peft_config = LoraConfig(r=LORA_R,lora_alpha=LORA_ALPHA,lora_dropout=0.05,target_modules="all-linear",task_type="CAUSAL_LM")
training_args = SFTConfig(output_dir=str(RUN_DIR),assistant_only_loss=True,max_length=MAX_LENGTH,packing=False,learning_rate=LEARNING_RATE,num_train_epochs=EPOCHS,per_device_train_batch_size=BATCH_SIZE,gradient_accumulation_steps=GRAD_ACCUMULATION,lr_scheduler_type="cosine",logging_steps=20,eval_strategy="epoch",save_strategy="epoch",fp16=True,bf16=False,seed=SEED,report_to="none",model_init_kwargs={"dtype":torch.float16})
save_json("training_config.json",training_args.to_dict())
print("RUN_DIR",RUN_DIR,flush=True)
try:
    trainer=SFTTrainer(model=BASE_MODEL,args=training_args,train_dataset=ds["train"],eval_dataset=ds["validation"],peft_config=peft_config,processing_class=tok,callbacks=[EvidenceCallback()])
    trainer.model.print_trainable_parameters()
    batch=next(iter(trainer.get_train_dataloader()))
    assert (batch["labels"] != -100).any().item(), "No assistant labels"
    save_json("label_mask_check.json",{"supervised_tokens":int((batch["labels"]!=-100).sum()),"ignored_tokens":int((batch["labels"]==-100).sum())})
    started=time.monotonic()
    train_result=trainer.train()
    ADAPTER_DIR=str(RUN_DIR / "final_adapter")
    trainer.save_model(ADAPTER_DIR)
    tok.save_pretrained(ADAPTER_DIR)
    trainer.save_state()
    save_json("train_metrics.json",train_result.metrics)
    save_json("train_log_history.json",trainer.state.log_history)
    provenance.update(status="training_saved",finished_utc=datetime.now(timezone.utc).isoformat(),elapsed_seconds=time.monotonic()-started,global_step=trainer.state.global_step,peak_gpu_bytes=torch.cuda.max_memory_allocated())
    save_json("provenance.json",provenance)
    print("TRAINING_SAVED",ADAPTER_DIR,train_result.metrics,flush=True)
except Exception:
    (RUN_DIR / "error.txt").write_text(traceback.format_exc())
    provenance["status"]="failed"
    save_json("provenance.json",provenance)
    raise


## 7. Save the adapter and confirm it is an adapter

In [ ]:
# The instrumented training cell already saved the adapter in the distinct RUN_DIR.
assert provenance["status"] == "training_saved"
assert Path(ADAPTER_DIR,"adapter_config.json").is_file()
assert Path(ADAPTER_DIR,"adapter_model.safetensors").is_file()
print(ADAPTER_DIR)


## 8. Smoke check on a few validation pairs

This only shows that the tuned model answers with a rewrite and what the fact checker says about it. It is not the evaluation: the validation split comes from the training sources, not from the golden test.

In [ ]:
from src.facts import compare_facts
from src.pipeline import clean_output

model = trainer.model
model.eval()
for row in ds["validation"].select(range(min(5, len(ds["validation"])))):
    messages = row["messages"]
    prompt = tok.apply_chat_template(messages[:-1], tokenize=False,
                                     add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    reply = clean_output(tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True))
    check = compare_facts(messages[1]["content"], reply)
    print("DIALECT :", messages[1]["content"])
    print("EXPECTED:", messages[2]["content"])
    print("MODEL   :", reply)
    print("        -> facts", "kept" if check.ok else f"DIFFER ({check.kind})", "\n")

## 9. Publish the adapter and its model card on the Hub

`login()` asks for your token. Do not paste the token into a cell and do not commit it. Fill in `docs/MODEL_CARD.md` with your real evaluation results once you have them and upload it again; until then the card says the results are pending.

In [ ]:
print("Publication disabled pending licence clearance. Keep adapters and GGUF files in the private team bundle.")


## 10. Merge the adapter and export GGUF files for Ollama

Ollama runs GGUF files. The adapter is merged into a full-precision copy of the base model and converted with the script from the llama.cpp repository (`q8_0`: 8-bit weights; a ready-made `Q8_0` file of this model on the Hub is listed at 2.17 GB).

The untuned base model is saved from the same half-precision copy, before the adapter is merged into it, and converted with the same script and the same setting. This is the baseline of the evaluation. The `qwen3:1.7b` build in the Ollama library is a 4-bit file (`Q4_K_M`), so comparing it with an 8-bit tuned file would mix the effect of quantisation with the effect of fine-tuning.

The published artefact that the project requires is still the adapter from step 9; the GGUF files are only how the two models are served locally.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

del trainer, model
torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16)
# The untuned model is saved first, from the same half-precision copy, so that the baseline goes through the same steps.
BASE_DIR = "outputs/base"
base.save_pretrained(BASE_DIR)
tok.save_pretrained(BASE_DIR)

# Merging changes `base` in place, which is why the untuned copy is saved before this line.
merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
MERGED_DIR = "outputs/merged"
merged.save_pretrained(MERGED_DIR)
tok.save_pretrained(MERGED_DIR)
print(sorted(os.listdir(MERGED_DIR)))

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip install -q sentencepiece protobuf
!python /content/llama.cpp/convert_hf_to_gguf.py outputs/merged --outfile outputs/{GGUF_NAME} --outtype q8_0
!ls -lh outputs/{GGUF_NAME}

In [ ]:
# The untuned base model saved in the cell above, converted with the same script and the same setting.
!python /content/llama.cpp/convert_hf_to_gguf.py {BASE_DIR} --outfile outputs/{BASE_GGUF_NAME} --outtype q8_0
!ls -lh outputs/{BASE_GGUF_NAME}

In [ ]:
print("Publication disabled pending licence clearance. Keep adapters and GGUF files in the private team bundle.")


## 11. Record the exact library versions

Commit `requirements-train.lock.txt` to the repository. It lists the versions that actually ran, which is what "pinned versions" has to mean for the training code.

In [ ]:
!pip freeze > requirements-train.lock.txt
!wc -l requirements-train.lock.txt

## 12. Evaluation after freeze and successful training

Use the same frozen V3-R1 file and a new results directory. Do not reuse old predictions, judgments or adapter names. First run the release preflight on the frozen checkout, record the data digest and code commit, and create the distinct Ollama models from the matched GGUF files.

```bash
python -m src.evaluate --golden data/golden_v3_r1_candidate.jsonl --base-model travel-v3-r1-base --tuned-model travel-v3-r1-lora --out results/v3_r1
python -m src.evaluate --golden data/golden_v3_r1_candidate.jsonl --from-predictions results/v3_r1 --judgments results/v3_r1/judgments.csv --out results/v3_r1
python -m src.safety_check --model travel-v3-r1-lora
```

Do not run until the active release preflight has passed. Copy the judgment template for two independent human raters. Do not mix historical V2.2 results into the V3-R1 directory.
